#### **EXTRAÇÃO E FILTRAGEM DE DADO FISCAIS DO SICONFI/TESOURO NACIONAL: 2013-2025**

**RESUMO**  
O script apresenta o roteiro para a extração e a filtragem de dados obtidos por meio da API do SICONFI, mantida pelo Tesouro Nacional, referentes aos estados brasileiros entre 2013 e 2025. O objetivo consiste em identificar as variáveis de interesse na Declaração de Contas Anuais (DCA) e exportar os resultados para um novo documento CSV. Essa abordagem evita o custo computacional atrelado ao uso de planilhas eletrônicas (principalmente Calc e Excel) e viabiliza uma alternativa mais leve e direta, recomendada para dataframes de grandes dimensões.

- **Fonte primária dos dados** disponível em <https://tesourotransparente.gov.br/consultas/consultas-siconfi/siconfi-api-de-dados-abertos>  
- **Base de dados completa em CSV no Google Drive (dca_estados.csv):** <https://drive.google.com/drive/folders/1GdsHXCOhOtgsQsi5TwyIiGrlYNiwGBss?usp=drive_link>

**Obs.:** O script para a extração e as opções de tratamento dos dados constam no arquivo 'data_stn_estados', disponível no repositório (<https://github.com/elsonr29/model_data_general>)

---------------------------------------------------
**AUTOR**: Prof. Dr. Elson Rodrigo de Souza Santos  
**E-MAIL**: [elson129@gmail.com] ou [elson.rodrigo@ufabc.edu.br]  
**REPOSITÓRIO**: https://github.com/elson29r/model_data_general  
**DATA**: Setembro de 2026

**Sumário**
1) Instalação dos pacotes
2) Preparação e filtro
3) Exportação
4) Visualização dos dados

**1) Instalação dos pacotes:**  

Pacotes utilizados:

- **Pandas** - análise e manutenção de em Python. O pacote fornece estruturas de dados eficientes, chamadas DataFrames, ideais para organizar painéis longitudinais, tratar séries temporais e preparar a matriz final de variáveis;

- **Matplotlib** - visualização mais antiga e mais usada do ecossistema Python;

- **Geopandas** - extensão do pandas para dados geoespaciais;

- **Plotly** - visualização interativa, os gráficos rodam no navegador ou no notebook com zoom, tooltip ao passar o mouse, e capacidade de ligar e desligar categorias clicando na legenda;

- **Seaborn** - construído sobre o matplotlib, mas com uma interface mais direta para gráficos estatísticos comuns, histogramas, boxplots, gráficos de dispersão com linha de tendência, comparação de distribuições entre grupos. Reduz a quantidade de código necessária para gráficos que, em matplotlib puro, exigiriam configuração manual mais longa.



In [ ]:
# Instalação

!pip install pandas matplotlib geopandas plotly seaborn

**2) Preparação e filtro**

Devido a dimensão do dataframe é pertinente filtrar as informações alvo. Para isso, utilizamos o pandas.

Para o projeto consideramos as variáveis de interesse:  
- Receita total  
- Receita corrente  
- Receita de capital   
- Operações de cŕedito  
- Despesa total  
- Despesa primária  
- Despesa corrente  
- Despesa juros e encargos  
- Despesas de capital  
- Investimento  
- Amortizações     
- Ativo (financeiro e não financeiro)
- Passivo (financeiro e não financeiro)
- Patrimônio Líquido

**Nota**: as informaçoes sobre cada estado são extremamente ricas e detalhas. A base de dados está em formato long que facilita a vizualização. Obviamente que pode ser usado o mesmo procedimento para escolher outras variáveis.


**Sumário**  
2.1 Carregar a base de dados 'dca_estados.csv'  
2.2 Verificação das informações e dimensões  
2.3 Filtragem e novo dataframe
 

In [ ]:
# 2.1 Carregar a base de dados 'dca_estados.csv'  

from pathlib import Path

import pandas as pd

dca = pd.read_csv("dca_estados.csv")

# Obs.: use o caminho da pasta em que salvou o arquivo 'dca_estados.csv'. Caso esteja na mesma pasta do script basta mante o mesmo caminho.


In [ ]:
# 2.2 Verificação das informações e dimensões  

print(dca.shape) # quantidade de linhas e colunas
print(dca.head()) # primeiros nomes
print(dca.dtypes) # tipos de variáveis

print(dca.columns.tolist()) # lista de colunas
print(dca.info()) # nomes, tipo de dado e contagem de não nulos por coluna

print(f"cod_conta: {dca['cod_conta'].nunique()} valores únicos")
print(f"conta: {dca['conta'].nunique()} valores únicos")

print(dca.loc[dca["anexo"].str.contains("I-C", na=False), "coluna"].unique())
print(dca.loc[dca["anexo"].str.contains("I-D", na=False), "coluna"].unique())

# Obs.: o dataframe apresenta cerca de 924 mil linhas e 113 colunas, dimensão que eleva o custo computacional e dificulta o uso de planilhas eletrônicas convencionais, como Excel e Calc.

In [ ]:
# 2.3 Filtragem e novo dataframe

# 2.3.1 mapas de código para nome padronizado

mapa_receita = {
    "TotalReceitas": "rec_total",
    "RO1.0.0.0.00.00.00": "rec_cor",
    "RO2.0.0.0.00.00.00": "rec_cap",
    "RO2.1.0.0.00.00.00": "rec_cap_cred",
}

mapa_despesa = {
    "TotalDespesas": "desp_total",
    "DO3.0.00.00.00.00": "desp_cor",
    "DO3.2.00.00.00.00": "desp_juros",
    "DO4.0.00.00.00.00": "desp_cap",
    "DO4.4.00.00.00.00": "desp_cap_invest",
    "DO4.6.00.00.00.00": "desp_cap_amort",
}

mapa_balanco_patrimonial = {
    "Ativo": "ati",
    "AtivoFinanceiro": "ati_fin",
    "AtivoPermanente": "ati_nfin",
    "AtivoReal": "ati_real",
    "Passivo": "pas",
    "PassivoFinanceiro": "pas_fin",
    "PassivoPermanente": "pas_nfin",
    "PassivoReal": "pas_real",
    "PatrimonioLiquido": "patri_liq",
}

# "Receitas Realizadas" (2013) e "Receitas Brutas Realizadas" (2014 em diante)
# identificam a mesma fase de execução, sob rótulos diferentes entre anos.
COLUNAS_RECEITA = ["Receitas Realizadas", "Receitas Brutas Realizadas"]
COLUNAS_DESPESA = ["Despesas Empenhadas"]



In [ ]:
# 2.3.2 funções de filtro

def filtrar_receita_despesa(df: pd.DataFrame, mapa_codigos: dict, colunas_alvo: list) -> pd.DataFrame:
    """Filtra por cod_conta e pela fase de execução em 'coluna',
    aceitando mais de um rótulo equivalente entre anos."""
    df = df.copy()
    df["cod_conta"] = df["cod_conta"].astype(str).str.strip()
    df["variavel"] = df["cod_conta"].map(mapa_codigos)

    filtrado = df[(df["variavel"].notna()) & (df["coluna"].isin(colunas_alvo))].copy()

    print(f"{filtrado.shape[0]} linhas casadas (cod_conta + coluna in {colunas_alvo}).")
    print(f"Variáveis encontradas: {sorted(filtrado['variavel'].unique())}")

    return filtrado


def filtrar_balanco(df: pd.DataFrame, mapa_codigos: dict) -> pd.DataFrame:
    """Filtra as linhas do balanço patrimonial por cod_conta."""
    df = df.copy()
    df["cod_conta"] = df["cod_conta"].astype(str).str.strip()
    df["variavel"] = df["cod_conta"].map(mapa_codigos)

    filtrado = df[df["variavel"].notna()].copy()

    print(f"{filtrado.shape[0]} linhas casadas com o mapa de códigos.")
    print(f"Variáveis encontradas: {sorted(filtrado['variavel'].unique())}")

    return filtrado


def pivotar_variaveis(df_filtrado: pd.DataFrame) -> pd.DataFrame:
    """Converte 'valor' para numérico (ponto decimal puro, sem
    separador de milhar) e transforma o formato longo filtrado
    em formato largo, uma coluna por variável."""
    df_filtrado = df_filtrado.copy()
    df_filtrado["valor"] = pd.to_numeric(df_filtrado["valor"], errors="coerce")

    return df_filtrado.pivot_table(
        index=["exercicio", "cod_ibge", "uf"],
        columns="variavel",
        values="valor",
        aggfunc="first",
    ).reset_index()




In [ ]:
# 2.3.3 Aplicação dos filtros e separação por anexo

receitas = dca[dca["anexo"].str.contains("I-C", na=False)]
despesas = dca[dca["anexo"].str.contains("I-D", na=False)]
patrimonial = dca[dca["anexo"].str.contains("I-AB", na=False)]

rec_filtrado = filtrar_receita_despesa(receitas, mapa_receita, COLUNAS_RECEITA)
desp_filtrado = filtrar_receita_despesa(despesas, mapa_despesa, COLUNAS_DESPESA)
bp_filtrado = filtrar_balanco(patrimonial, mapa_balanco_patrimonial)


**3) Formatação e Exportação**

Formatação dos dados em novo dataframe csv e exporta

**Sumário**  
3.1 Formatação  
3.2 Pivot para formato largo  
3.3 Junção e variáveis construídas  
3.4 Exportação em formato largo  

In [ ]:
# 3.1 formatação

colunas_conferencia = [
    "exercicio", "cod_ibge", "uf", "anexo", "coluna",
    "cod_conta", "conta", "variavel", "valor",
]

base_longa = pd.concat([rec_filtrado, desp_filtrado, bp_filtrado], ignore_index=True)
base_longa = base_longa[colunas_conferencia]
base_longa.to_csv("dca_variaveis_long.csv", index=False)
print(f"Formato longo exportado: {base_longa.shape[0]} linhas.")

In [ ]:
# 3.2 pivot para formato largo

base_rec = pivotar_variaveis(rec_filtrado)
base_desp = pivotar_variaveis(desp_filtrado)
base_bp = pivotar_variaveis(bp_filtrado)

base_bp["ati"] = base_bp["ati"].fillna(base_bp["ati_fin"] + base_bp["ati_nfin"])
base_bp["pas"] = base_bp["pas"].fillna(base_bp["pas_fin"] + base_bp["pas_nfin"])
base_bp["patri_liq"] = base_bp["patri_liq"].fillna(base_bp["ati"] - base_bp["pas"])


In [ ]:
# 3.3 junção e variáveis construídas

base_rd = base_rec.merge(base_desp, on=["exercicio", "cod_ibge", "uf"], how="outer")

base_rd["desp_pri"] = base_rd["desp_total"] - base_rd["desp_juros"] - base_rd["desp_cap_amort"]
base_rd["desp_corr"] = base_rd["desp_cor"] - base_rd["desp_juros"]
base_rd["rec_cap_out"] = base_rd["rec_cap"] - base_rd["rec_cap_cred"]

base_final = base_rd.merge(base_bp, on=["exercicio", "cod_ibge", "uf"], how="outer")

In [ ]:
# 3.4 exportação em formato largo, para análise ---

base_final.to_csv("dca_variaveis_wide.csv", index=False)
print(f"Formato largo exportado: {base_final.shape}")
print(base_final.isna().sum())

**4) Visualização dos dados**

Exporamos os dados obtidos após a filtragem.

In [ ]:
import plotly.express as px

fig = px.line(
    base_final,
    x="exercicio", y="desp_pri", color="uf",
    title="Despesa primária por estado, 2013-2025",
)
fig.show()

In [ ]:
fig = px.scatter(
    base_final,
    x="rec_total", y="desp_total", color="uf",
    animation_frame="exercicio",
    title="Receita total vs despesa total, por estado e ano",
)
fig.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(data=base_final[base_final["exercicio"] == 2023], x="uf", y="desp_juros")
plt.xticks(rotation=90)
plt.show()

In [ ]:
sns.lmplot(data=base_final, x="rec_cor", y="desp_cor", hue="exercicio", height=6)

In [ ]:
colunas_numericas = ["rec_total", "rec_cor", "desp_total", "desp_pri", "ati", "pas", "patri_liq"]
sns.heatmap(base_final[colunas_numericas].corr(), annot=True, cmap="coolwarm")

In [ ]:
# Mapa coroplético do Brasil por estado — Patrimônio Líquido

import os

# necessário caso o .shx do shapefile não tenha extraído corretamente
os.environ["SHAPE_RESTORE_SHX"] = "YES"

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# 1 malha geográfica dos estados (IBGE, BR_UF_2022)
# origem: https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/
#         malhas_municipais/municipio_2022/Brasil/BR/BR_UF_2022.zip

malha = gpd.read_file("BR_UF_2022/BR_UF_2022.shp")
malha["CD_UF"] = malha["CD_UF"].astype(int)

# Obs.: usamos para gerar os gráficos seguintes

In [ ]:
# 2. base fiscal já filtrada (dca_variaveis_wide.csv)

dados_2025 = base_final[base_final["exercicio"] == 2025]
malha_dados = malha.merge(dados_2025, left_on="CD_UF", right_on="cod_ibge")

fig, ax = plt.subplots(figsize=(10, 8))
malha_dados.plot(
    column="patri_liq",
    legend=True,
    cmap="viridis",
    edgecolor="black",
    linewidth=0.3,
    ax=ax,
)
ax.set_title("Patrimônio líquido por estado, 2025")
ax.axis("off")
plt.show()

In [ ]:
# Investimento em relação a despesa primária despesa primária (despesa primária já exclui juros e amortizações)

base_final["invest_sobre_desp_pri"] = base_final["desp_cap_invest"] / base_final["desp_pri"]

print(base_final[["exercicio", "uf", "desp_cap_invest", "desp_pri", "invest_sobre_desp_pri"]].head(10))
print(base_final["invest_sobre_desp_pri"].describe())

In [ ]:
# 3. relação investimento/despesa primária, ano de 2025

dados_2025 = base_final[base_final["exercicio"] == 2025]
malha_dados = malha.merge(dados_2023, left_on="CD_UF", right_on="cod_ibge")

fig, ax = plt.subplots(figsize=(10, 8))
malha_dados.plot(
    column="invest_sobre_desp_pri",
    legend=True,
    cmap="viridis",
    edgecolor="black",
    linewidth=0.3,
    ax=ax,
)
ax.set_title("Investimento / Despesa Primária por estado, 2023")
ax.axis("off")
plt.show()

In [ ]:
# Relação entre despesa corrente e despesa primária, 2025

base_final["desp_corr_sobre_desp_pri"] = base_final["desp_corr"] / base_final["desp_pri"]

dados_2025 = base_final[base_final["exercicio"] == 2025]
malha_dados = malha.merge(dados_2025, left_on="CD_UF", right_on="cod_ibge")

fig, ax = plt.subplots(figsize=(10, 8))
malha_dados.plot(
    column="desp_corr_sobre_desp_pri",
    legend=True,
    cmap="viridis",
    edgecolor="black",
    linewidth=0.3,
    ax=ax,
)
ax.set_title("Despesa Corrente / Despesa Primária por estado, 2025")
ax.axis("off")
plt.show()

In [ ]:
# despesa corrente e investimento versus despesa primária

import plotly.express as px

comparacao = base_final[base_final["uf"].isin(["PR", "SC", "RS"])]
fig = px.line(
    comparacao, x="exercicio", y="invest_sobre_desp_pri", color="uf",
    title="Investimento / Despesa Primária — Sul do Brasil",
)
fig.show()

In [ ]:
import plotly.graph_objects as go

pr = base_final[base_final["uf"] == "PR"].sort_values("exercicio")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=pr["exercicio"], y=pr["desp_corr_sobre_desp_pri"],
    mode="lines+markers", name="Despesa Corrente / Despesa Primária",
))
fig.add_trace(go.Scatter(
    x=pr["exercicio"], y=pr["invest_sobre_desp_pri"],
    mode="lines+markers", name="Investimento / Despesa Primária",
))

fig.update_layout(
    title="Composição da despesa primária — Paraná, 2013-2025",
    xaxis_title="Ano",
    yaxis_title="Proporção da despesa primária",
    hovermode="x unified",
)
fig.show()

**FIM**